In [ ]:
%load_ext autoreload
%autoreload 2

# Eva Data

This notebook builds a `MultiDataset` containing exactly one `ZarrDataset`, loads one batch, visualizes one image with `mediapy`, and prints the rest of the batch.

In [ ]:
from pathlib import Path

import cv2
import imageio_ffmpeg
import mediapy as mpy
import numpy as np
import torch

from egomimic.rldb.embodiment.eva import Eva
from egomimic.rldb.embodiment.human import Aria
from egomimic.rldb.zarr.zarr_dataset_multi import MultiDataset, ZarrDataset
from egomimic.rldb.zarr.zarr_dataset_multi import S3EpisodeResolver
from egomimic.rldb.filters import DatasetFilter
from egomimic.utils.aws.aws_data_utils import load_env

# Ensure mediapy can find an ffmpeg executable in this environment
mpy.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())

In [ ]:
TEMP_DIR = "/coc/flash7/scratch/egoverseDebugDatasets/egoverseS3DatasetTest/" # replace with your own temp directory for caching S3 data
load_env()

In [ ]:
key_map = Eva.get_keymap(keymap_mode="cartesian")
transform_list = Eva.get_transform_list(mode="cartesian")

resolver = S3EpisodeResolver(
    TEMP_DIR, key_map=key_map, transform_list=transform_list
)
filters = DatasetFilter(
    filter_lambdas=[
        "lambda row: row['episode_hash'] in {'2026-01-03-02-16-02-519000'}"
    ]
)
multi_ds = MultiDataset._from_resolver(
    resolver, filters=filters, sync_from_s3=True, mode="total"
)

loader = torch.utils.data.DataLoader(multi_ds, batch_size=1, shuffle=False)

In [ ]:
# Separate YPR visualization preview
for batch in loader:
    vis_ypr = Eva.viz_transformed_batch(batch, mode="axes")
    mpy.show_image(vis_ypr)
    break

In [ ]:
images = []
for i, batch in enumerate(loader):
    vis = Eva.viz_transformed_batch(batch, mode="traj+rotation")
    images.append(vis)
    if i > 10:
        break

mpy.show_video(images, fps=30)

## Human Datasets
Mecka, Scale and Aria all use different key_maps 

In [ ]:
from egomimic.rldb.zarr.zarr_dataset_multi import LocalEpisodeResolver
intrinsics_key = "base"

key_map = Aria.get_keymap(keymap_mode="cartesian")
transform_list = Aria.get_transform_list(mode="cartesian")

TEMP_DIR = '/coc/flash7/scratch/egoverseDebugDatasets/egoverseS3DatasetTest/'
resolver = LocalEpisodeResolver(
    TEMP_DIR,
    key_map=key_map,
    transform_list=transform_list,
)

filters = DatasetFilter(
    filter_lambdas=[
        "lambda row: row['episode_hash'] in {'2026-05-01-02-52-58-000000'}"
    ]
)

cloudflare_ds = MultiDataset._from_resolver(
    resolver, filters=filters, sync_from_s3=False, mode="total"
)

loader = torch.utils.data.DataLoader(cloudflare_ds, batch_size=1, shuffle=False)

In [ ]:
batch = next(iter(loader))

In [ ]:
batch.keys()

In [ ]:
ims = []
for i, batch in enumerate(loader):
    vis = LightWheel.viz_transformed_batch(batch, mode="traj")
    ims.append(vis)
    # mpy.show_image(vis)

    # for k, v in batch.items():
    #     print(f"{k}: {tuple(v.shape)}")
    
    if i > 10:
        break

mpy.show_video(ims, fps=30)


In [ ]:
# Aria YPR video (same data loop, YPR overlay)
ims_ypr = []
for i, batch in enumerate(loader):
    vis_ypr = Aria.viz_transformed_batch(batch, mode="axes")
    ims_ypr.append(vis_ypr)
    if i > 20:
        break

mpy.show_video(ims_ypr, fps=30)

In [ ]:
key_map = Aria.get_keymap(mode="keypoints")
transform_list = Aria.get_transform_list(mode="keypoints_headframe_ypr")

resolver = S3EpisodeResolver(
    TEMP_DIR,
    key_map=key_map,
    transform_list=transform_list,
)

cloudflare_ds = MultiDataset._from_resolver(
    resolver, filters=filters, sync_from_s3=True, mode="total"
)

loader = torch.utils.data.DataLoader(cloudflare_ds, batch_size=1, shuffle=False)

In [ ]:
from egomimic.rldb.embodiment.human import _build_aria_keypoints_revert_eef_frame_transform_list
ims_keypoints = []
revert_transform_list = _build_aria_keypoints_revert_eef_frame_transform_list(is_quat=False)
for i, batch in enumerate(loader):
    vis_keypoints = Aria.viz_transformed_batch(batch, mode="keypoints", viz_batch_key="actions_keypoints")
    ims_keypoints.append(vis_keypoints)
    if i > 20:
        break

mpy.show_video(ims_keypoints, fps=30)

## Aria Gaze

In [ ]:
from egomimic.rldb.zarr.zarr_dataset_multi import MultiDataset, ZarrDataset
import mediapy as mpy
import numpy as np
import torch
from pathlib import Path

keymap_gaze = Aria.get_keymap(keymap_mode="gaze")

filters = DatasetFilter(
    filter_lambdas=[
        "lambda row: row['episode_hash'] in {'2026-05-01-02-52-58-000000'}"
    ]
)
resolver = S3EpisodeResolver(
    TEMP_DIR, key_map=keymap_gaze
)
cloudflare_ds = MultiDataset._from_resolver(
    resolver, filters=filters, sync_from_s3=True, mode="total"
)

loader = torch.utils.data.DataLoader(cloudflare_ds, batch_size=1, shuffle=False)

In [ ]:
loader

In [ ]:
batch = next(iter(loader))
print(batch.keys())

In [ ]:
ims_gaze = []
for i, batch in enumerate(loader):
    vis_gaze = Aria.viz_transformed_batch(
        batch,
        mode="gaze",
        viz_batch_key="eye_gaze",
        image_key="images.front_1",
    )
    ims_gaze.append(vis_gaze)
    if i > 20:
        break
mpy.show_video(ims_gaze, fps=30)

# Annotation Viz

In [ ]:
from egomimic.rldb.embodiment.human import Scale
from egomimic.rldb.filters import DatasetFilter
intrinsics_key = "base"

key_map = Scale.get_keymap(mode="cartesian", annotations=True)
transform_list = None

resolver = S3EpisodeResolver(
    folder_path = "/coc/flash7/scratch/egoverseDebugDatasets/egoverseS3DatasetTest/",
    key_map=key_map,
    transform_list=transform_list,
)

filters = DatasetFilter(
    filter_lambdas=[
        "lambda row: row['episode_hash'] in {'2026-03-16-01-22-26-448000'}"
    ]
)

cloudflare_ds = MultiDataset._from_resolver(
    resolver, filters=filters, sync_from_s3=True, mode="total"
)

loader = torch.utils.data.DataLoader(cloudflare_ds, batch_size=1, shuffle=False)

In [ ]:
batch = next(iter(loader))
batch['annotations']

In [ ]:
ims_annotations = []
for i, batch in enumerate(loader):
    vis = Scale.viz_transformed_batch(batch, mode="annotations", viz_batch_key="annotations")
    ims_annotations.append(vis)
    if i > 10:
        break
mpy.show_video(ims_annotations, fps=30)